<a href="https://colab.research.google.com/github/anyuanay/INFO213/blob/main/INFO213_Week6_KNN_lecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 213: Data Science Programming 2
___

### KNN
___


**Objectives:**
- Describe neighbors in terms of notions of distances
- Explain the issues with dimensionality
- Define and compute distances for multi-dimensional data

# Load Data
The wine data set:
- The data is the results of a chemical analysis of wines grown in the same region in Italy by three different cultivators. There are thirteen different measurements taken for different constituents found in the three types of wine.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# load the wine data from scikit learn data sets
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)

df_wine = data.frame

In [ ]:
# rename the column names
df_wine.columns = ['Alcohol', 'Malic acid', 'Ash',
                   'Alcalinity of ash', 'Magnesium', 'Total phenols',
                   'Flavanoids', 'Nonflavanoid phenols', 'Proanthocyanins',
                   'Color intensity', 'Hue', 'OD280/OD315 of diluted wines',
                   'Proline', 'Class_label']

print('Class labels', np.unique(df_wine['Class_label']))

Class labels [0 1 2]


In [ ]:
df_wine.shape

(178, 14)

# Split the data into trainning and test set

In [ ]:
from sklearn.model_selection import train_test_split

X, y = df_wine.iloc[:, :-1].values, df_wine.iloc[:, -1].values

X_train, X_test, y_train, y_test =\
    train_test_split(X, y,
                     test_size=0.3,
                     random_state=0,
                     stratify=y)

In [ ]:
X.shape, y.shape

((178, 13), (178,))

# Bringing features onto the same scale

In [ ]:
# Min-max scaler
from sklearn.preprocessing import MinMaxScaler

mms = MinMaxScaler()
X_train_norm = mms.fit_transform(X_train)
X_test_norm = mms.transform(X_test)

In [ ]:
# Standard scaler
from sklearn.preprocessing import StandardScaler

stdsc = StandardScaler()
X_train_std = stdsc.fit_transform(X_train)
X_test_std = stdsc.transform(X_test)

# KNN classifier:
- Instance-based learning algorithm, It doesn't build an explicit model but stores the training data.

- A new data point is classified by a majority vote of its k nearest neighbors in the training dataset.

- Commonly uses Euclidean distance (but others like Manhattan, Minkowski, or cosine similarity can be used).

- Small k → more sensitive to noise.

- Large k → more stable but may include distant or irrelevant neighbors.

- Feature scaling is important: KNN is sensitive to the scale of the features, so normalization/standardization is often necessary.

- Curse of dimensionality: Performance can degrade in high-dimensional spaces due to sparse data.

- Works well for small datasets: Not suitable for large datasets because of high computation cost during prediction.

## The Model


KNN makes no mathematical assumptions, and it doesn’t require any sort of heavy machinery. The only things it requires are:
- Some notion of distance
- An assumption that points that are close to one another are similar

In general, data are represented as vectors. For a given data $x$,  we find the nearest $k$ neighbors to predict $x$'s label.

Let’s say we’ve picked a number $k$ like 3 or 5. Then when we want to classify some new data point, we find the $k$ nearest labeled points and let them vote on the new output.

![](https://raw.githubusercontent.com/anyuanay/INFO213/main/knn.png)

To do this, we’ll need a function that counts votes. One possibility is:

```
def raw_majority_vote(labels):
    votes = Counter(labels)
    winner, _ = votes.most_common(1)[0]
    return winner
```

In [ ]:
labels = ['r', 'r', 'g', 'y', 'y']

In [ ]:
raw_majority_vote(labels)

'r'

What happens if there are several labels having equal majority votes? We need a strategy to break the tie. There are several options:
* Pick one of the winners at random.
* Weight the votes by distance and pick the weighted winner.
* Reduce k until we find a unique winner.

The following implements the third:

```
from collections import Counter

def majority_vote(labels):
    """
    Perform a majority vote among ordered labels.
    Assumes labels are sorted from nearest to farthest (e.g., in k-NN).
    """

    # Count how many times each label appears
    vote_counts = Counter(labels)

    # Find the label with the highest vote and its count
    # most_common(1) returns a list like [(label, count)]
    winner, winner_count = vote_counts.most_common(1)[0]

    # Count how many labels are tied with the highest vote count
    num_winners = len([
        count
        for count in vote_counts.values()
        if count == winner_count
    ])

    # If only one label has the highest count, it's the clear winner
    if num_winners == 1:
        return winner

    else:
        # Tie-breaking strategy:
        # Remove the farthest label (last element) and retry
        # This works because labels are ordered by distance
        return majority_vote(labels[:-1])
```

With this, we can create a KNN classifier:

```
from scipy.spatial import distance
def knn_classify(k, labeled_points, new_point):
    """each labeled point should be a pair (point, label)"""

    # order the labeled points from nearest to farthest
    by_distance = sorted(labeled_points,
                         key=lambda point_label: distance.euclidean(point_label[0], new_point))

    # find the labels for the k closest
    k_nearest_labels = [label for _, label in by_distance[:k]]

    # and let them vote
    return majority_vote(k_nearest_labels)
```

### Apply the knn_classify to the wine data

```
# Combine features and labels into (point, label) pairs
labeled_points = list(zip(X_train_std, y_train))
```

```
# Choose k
k = 5

# Predict labels for test data
y_pred = [knn_classify(k, labeled_points, x) for x in X_test_std]
```

```
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)
```

## Use Scikit Learn KNN Classifier

```
from sklearn.neighbors import KNeighborsClassifier
```

```
from sklearn.metrics import accuracy_score
```

```
# Accuracy before feature selection
knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train_std, y_train)

y_pred = knn.predict(X_test_std)

accuracy_score(y_test, y_pred)
```

# Retrieval Practice

## The Curse of Dimensionality
k-nearest neighbors runs into trouble in higher dimensions thanks to the “curse of dimensionality,” which boils down to the fact that high-dimensional spaces are vast. Points in high-dimensional spaces tend not to be close to one another at all. One way to see this is by randomly generating pairs of points in the d-dimensional “unit cube”
in a variety of dimensions, and calculating the distances between them.

Let us define a function to compute a list (len = num_pairs) of distances between two points with dimensions $dim$:

```
from scipy.spatial import distance
import numpy as np
import random

def random_distances(dim, num_pairs):
    return [distance.euclidean(np.random.rand(dim), np.random.rand(dim))
            for _ in range(num_pairs)]
```

For a range of dimensions between 1 and 101 with stride 5, compute 10000 distances:

```
dimensions = range(1, 101, 5)

avg_distances = []
min_distances = []

random.seed(0)
for dim in dimensions:
     distances = random_distances(dim, 10000)  # 10,000 random pairs
     avg_distances.append(np.mean(distances))     # track the average
     min_distances.append(min(distances))      # track the minimum
     #print(dim, min(distances), mean(distances), min(distances) / mean(distances))```

Plot the ratio between mininum distances and averge distances as the dimension increases:

```
plt.plot(list(dimensions), list(np.array(min_distances) / np.array(avg_distances)))```

- In low-dimensional data sets, the closest points tend to be much closer than average.
- But two points are close only if they’re close in every dimension, and every extra dimension—even if just noise—is another opportunity for each point to be further away from every other point.
- When you have a lot of dimensions, it’s likely that the closest points aren’t much closer than average, which means that two points being close doesn’t mean very much (unless there’s a lot of structure in your data that makes it behave as if it were much lower-dimensional).

- A different way of thinking about the problem involves the sparsity of higherdimensional spaces.

- If you pick 50 random numbers between 0 and 1, you’ll probably get a pretty good sample of the unit interval.
- If you pick 50 random points in the unit square, you’ll get less coverage.

- And in three dimensions less still.

- So if you’re trying to use nearest neighbors in higher dimensions, it’s probably a good idea to do some kind of dimensionality reduction first.

## Use Scikit Learn SequentialFeatureSelector

```
from sklearn.feature_selection import SequentialFeatureSelector
```

```
knn = KNeighborsClassifier(n_neighbors=5)
```

```
# Sequential Feature Selector (SFS)
# ---------------------------------
# This creates a feature selection wrapper around the KNN model.
# It will iteratively choose a subset of features that gives the best model performance.

sfs = SequentialFeatureSelector(knn, n_features_to_select=6)
```

```
sfs.fit(X_train_std, y_train)
```

```
sfs.support_
```

```
# selected features
selected_columns = df_wine.columns[:-1][sfs.support_]
selected_columns
```

```
# Accuracy after feature selection
knn = KNeighborsClassifier(n_neighbors=5)

X_train_std_selected = X_train_std[:, sfs.support_]
X_train_std.shape, X_train_std_selected.shape
```

```
knn.fit(X_train_std_selected, y_train)

y_pred = knn.predict(X_test_std[:, sfs.support_])

accuracy_score(y_test, y_pred)
```

- As we can see the accuracy of the KNN classifier improved on the validation dataset as we reduced the number of features.
- It is likely due to a decrease in the curse of dimensionality.

## KNN is Sensitive to Scale of Data
- Different scales may result in different distances of data.
- Imagine that you have a data set consisting of the heights and weights of hundreds of data scientists, and that you are trying to identify clusters of body sizes.
- Consider the people listed following table:

```
df = pd.DataFrame({"Person":['A', 'B', 'C'], "height (cm)":[160, 170.2, 177.8], "weight (pounds)":[150, 160, 171], "height (inches)":[63, 67, 70]})
df
```

In [ ]:
df = pd.DataFrame({"Person":['A', 'B', 'C'], "height (cm)":[160, 170.2, 177.8], "weight (pounds)":[150, 160, 171], "height (inches)":[63, 67, 70]})
df

,Person,height (cm),weight (pounds),height (inches)
0,A,160.0,150,63
1,B,170.2,160,67
2,C,177.8,171,70


The distances between the pairs of persons in terms of height (inches) and weight (pounds) are:

```
from scipy.spatial import distance
print("A to B: " + str(distance.euclidean(df.iloc[0, 2:], df.iloc[1, 2:])))
print("A to C: " + str(distance.euclidean(df.iloc[0, 2:], df.iloc[2, 2:])))
print("B to C: " + str(distance.euclidean(df.iloc[1, 2:], df.iloc[2, 2:])))```

In [ ]:
from scipy.spatial import distance
print("A to B: " + str(distance.euclidean(df.iloc[0, 2:], df.iloc[1, 2:])))
print("A to C: " + str(distance.euclidean(df.iloc[0, 2:], df.iloc[2, 2:])))
print("B to C: " + str(distance.euclidean(df.iloc[1, 2:], df.iloc[2, 2:])))

A to B: 10.770329614269007
A to C: 22.135943621178654
B to C: 11.40175425099138


The distnace between the pairs of persons in terms of height (cm) and weight (pounts) are:

```
print("A to B: " + str(distance.euclidean(df.iloc[0, 1::2], df.iloc[1, 1::2])))
print("A to C: " + str(distance.euclidean(df.iloc[0, 1::2], df.iloc[2, 1::2])))
print("B to C: " + str(distance.euclidean(df.iloc[1, 1::2], df.iloc[2, 1::2])))```

In [ ]:
print("A to B: " + str(distance.euclidean(df.iloc[0, 1::2], df.iloc[1, 1::2])))
print("A to C: " + str(distance.euclidean(df.iloc[0, 1::2], df.iloc[2, 1::2])))
print("B to C: " + str(distance.euclidean(df.iloc[1, 1::2], df.iloc[2, 1::2])))

A to B: 10.95627673984186
A to C: 19.126944345608383
B to C: 8.17067928632622


- The nearest point to B is different using different units.
- Obviously it’s problematic if changing units can change results like this.
- For this reason, when dimensions aren’t comparable with one another, we will sometimes rescale our data so that each dimension has mean 0 and standard deviation 1.
- This effectively gets rid of the units, converting each dimension to "standard deviations from the mean."

In [ ]:
df_data = df.set_index('Person')
df_scaled = (df_data - df_data.mean(axis = 0)) / df_data.std(axis = 0)
df_scaled

,height (cm),weight (pounds),height (inches)
Person,,,
A,-1.044980,-0.983755,-1.044074
B,0.097034,-0.031734,0.094916
C,0.947946,1.015489,0.949158


```
df_scaled.mean(axis = 0)
```

In [ ]:
df_scaled.mean()

,0
height (cm),-9.992007e-16
weight (pounds),-8.881784e-16
height (inches),-1.332268e-15


```
df_scaled.std(axis = 0)
```

In [ ]:
df_scaled.std(axis = 0)

,0
height (cm),1.0
weight (pounds),1.0
height (inches),1.0


```
# the distances using height in inches:
from scipy.spatial import distance
print("A to B: " + str(distance.euclidean(df_scaled.iloc[0, 1:], df_scaled.iloc[1, 1:])))
print("A to C: " + str(distance.euclidean(df_scaled.iloc[0, 1:], df_scaled.iloc[2, 1:])))
print("B to C: " + str(distance.euclidean(df_scaled.iloc[1, 1:], df_scaled.iloc[2, 1:])))```

In [ ]:
# the distances using height in inches:
from scipy.spatial import distance
print("A to B: " + str(distance.euclidean(df_scaled.iloc[0, 1:], df_scaled.iloc[1, 1:])))
print("A to C: " + str(distance.euclidean(df_scaled.iloc[0, 1:], df_scaled.iloc[2, 1:])))
print("B to C: " + str(distance.euclidean(df_scaled.iloc[1, 1:], df_scaled.iloc[2, 1:])))

A to B: 1.4844668093876097
A to C: 2.8231103104442656
B to C: 1.3514460651057632


```
# the distances using height in cm:
from scipy.spatial import distance
print("A to B: " + str(distance.euclidean(df_scaled.iloc[0, 0::2], df_scaled.iloc[1, 0::2])))
print("A to C: " + str(distance.euclidean(df_scaled.iloc[0, 0::2], df_scaled.iloc[2, 0::2])))
print("B to C: " + str(distance.euclidean(df_scaled.iloc[1, 0::2], df_scaled.iloc[2, 0::2])))```

In [ ]:
# the distances using height in cm:
from scipy.spatial import distance
print("A to B: " + str(distance.euclidean(df_scaled.iloc[0, 0::2], df_scaled.iloc[1, 0::2])))
print("A to C: " + str(distance.euclidean(df_scaled.iloc[0, 0::2], df_scaled.iloc[2, 0::2])))
print("B to C: " + str(distance.euclidean(df_scaled.iloc[1, 0::2], df_scaled.iloc[2, 0::2])))


A to B: 1.6129142931621308
A to C: 2.818639081896178
B to C: 1.205728497183663


# Retrieval Practice